
## **Problem Statement:**
Customer churn, the rate at which customers stop doing business with a company, is a significant challenge in the telecommunications industry.
High churn rates can lead to substantial revenue losses and increased costs for acquiring new customers.
Understanding the factors that contribute to churn and accurately predicting which customers are likely to churn are crucial for implementing effective retention strategies.

## Objective:
The objective of this project is to build an end-to-end machine learning pipeline to predict customer churn using the provided Telco Churn Dataset.
The pipeline will involve data preprocessing, model development (using Logistic Regression and Random Forest), hyperparameter tuning, and model evaluation to identify customers at risk of churning.

## **Dependencies and libraries**

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
# Load the Telco Churn dataset from the specified path into a pandas DataFrame

df = pd.read_csv('/content/sample_data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
# Display the first 5 rows of the DataFrame to get a preview of the data
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## **Data Preprocessing**


In [ ]:
# Inspect for missing values
print("Missing values before handling:")
print(df.isnull().sum())

# The 'TotalCharges' column has some missing values represented as ' '.
# Convert 'TotalCharges' to numeric, coercing errors to NaN.
# This is a necessary step to handle non-numeric entries that represent missing data.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Handle missing values in 'TotalCharges' by dropping rows with NaN
# Dropping rows is a simple approach when the number of missing values is small.
df.dropna(subset=['TotalCharges'], inplace=True)

print("\nMissing values after handling:")
print(df.isnull().sum())

# Separate features (X) and target variable (y)
# 'Churn' is the target variable we want to predict.
X = df.drop('Churn', axis=1)
y = df['Churn']

# Identify categorical and numerical columns
# Exclude 'customerID' as it's an identifier and not a feature used for modeling directly.
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("\nCategorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Missing values before handling:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Missing values after handling:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0

## **Preprocessing Pipeline Creation**

In [ ]:
# Define preprocessing steps for numerical and categorical features
# Exclude 'customerID' from the categorical columns for preprocessing as it's not a predictive feature.
categorical_features_for_pipeline = [col for col in categorical_cols if col != 'customerID']

# Create a ColumnTransformer to apply different preprocessing steps to different column types.
# 'num': Applies StandardScaler to numerical columns for scaling.
# 'cat': Applies OneHotEncoder to categorical columns for encoding, ignoring unknown categories during prediction.
# 'remainder': 'passthrough' keeps any columns not explicitly listed in transformers (like customerID, although we will drop it later).
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_for_pipeline)
    ],
    remainder='passthrough'
)

# Create a preprocessing pipeline. This pipeline encapsulates the ColumnTransformer.
# This makes it easy to apply the same preprocessing steps consistently.
preprocess_pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## **Model Development and Training**

In [ ]:
# Drop the 'customerID' column from the feature set X as it's not needed for training.
# The ColumnTransformer was set to passthrough, so we explicitly drop it here before passing X to the pipeline.
X = X.drop('customerID', axis=1)

# Create the Logistic Regression pipeline.
# This pipeline first applies the preprocessing steps defined in 'preprocess_pipeline'
# and then trains a Logistic Regression classifier.
lr_pipeline = Pipeline(steps=[('preprocessor', preprocess_pipeline['preprocessor']),
                              ('classifier', LogisticRegression())])

# Create the Random Forest pipeline.
# Similar to the LR pipeline, this applies preprocessing followed by a Random Forest classifier.
# A random_state is set for reproducibility.
rf_pipeline = Pipeline(steps=[('preprocessor', preprocess_pipeline['preprocessor']),
                              ('classifier', RandomForestClassifier(random_state=42))])

# Train the Logistic Regression pipeline
# The fit method applies the preprocessing and then trains the model.
lr_pipeline.fit(X, y)

# Train the Random Forest pipeline
# The fit method applies the preprocessing and then trains the model.
rf_pipeline.fit(X, y)

print("Logistic Regression and Random Forest pipelines created and trained successfully after dropping customerID.")

Logistic Regression and Random Forest pipelines created and trained successfully after dropping customerID.


## **Hyperparameter Tuning**

In [ ]:
# Define parameter grid for Logistic Regression
# 'classifier__C' is the inverse of regularization strength. Smaller values specify stronger regularization.
param_grid_lr = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
}

# Define parameter grid for Random Forest
# 'classifier__n_estimators' is the number of trees in the forest.
# 'classifier__max_depth' is the maximum depth of the trees. None means nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
param_grid_rf = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30],
}

# Instantiate GridSearchCV for Logistic Regression
# cv=5 for 5-fold cross-validation, scoring='accuracy' as the evaluation metric, n_jobs=-1 to use all available CPU cores.
grid_search_lr = GridSearchCV(lr_pipeline, param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)

# Instantiate GridSearchCV for Random Forest
# cv=5 for 5-fold cross-validation, scoring='accuracy' as the evaluation metric, n_jobs=-1 to use all available CPU cores.
grid_search_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)

# Fit GridSearchCV to the data for both models
# This performs the hyperparameter search using cross-validation.
print("Starting GridSearchCV for Logistic Regression...")
grid_search_lr.fit(X, y)
print("GridSearchCV for Logistic Regression completed.")

print("Starting GridSearchCV for Random Forest...")
grid_search_rf.fit(X, y)
print("GridSearchCV for Random Forest completed.")

# Print the best parameters found by GridSearchCV
print("\nBest parameters for Logistic Regression:", grid_search_lr.best_params_)
print("Best parameters for Random Forest:", grid_search_rf.best_params_)

Starting GridSearchCV for Logistic Regression...
GridSearchCV for Logistic Regression completed.
Starting GridSearchCV for Random Forest...
GridSearchCV for Random Forest completed.

Best parameters for Logistic Regression: {'classifier__C': 100}
Best parameters for Random Forest: {'classifier__max_depth': 10, 'classifier__n_estimators': 200}


## **Model Evaluation**



In [ ]:
# Get the best estimators found by GridSearchCV
# These are the pipelines with the optimal hyperparameters.
best_lr_model = grid_search_lr.best_estimator_
best_rf_model = grid_search_rf.best_estimator_

# Make predictions on the feature data X using the best models
# Note: This evaluation is on the training data itself. For a more robust evaluation,
# a separate test set should ideally be used.
y_pred_lr = best_lr_model.predict(X)
y_pred_rf = best_rf_model.predict(X)

# Calculate and print evaluation metrics for Logistic Regression
print("--- Logistic Regression Metrics (on Training Data) ---")
# Accuracy: Proportion of correctly classified instances.
print(f"Accuracy: {accuracy_score(y, y_pred_lr):.4f}")
# Precision: Ability of the classifier not to label as positive a negative sample.
# 'pos_label='Yes'' specifies that 'Yes' is the positive class for churn.
print(f"Precision: {precision_score(y, y_pred_lr, pos_label='Yes'):.4f}")
# Recall: Ability of the classifier to find all the positive samples.
print(f"Recall: {recall_score(y, y_pred_lr, pos_label='Yes'):.4f}")
# F1-Score: Harmonic mean of precision and recall.
print(f"F1-Score: {f1_score(y, y_pred_lr, pos_label='Yes'):.4f}")

# Calculate and print evaluation metrics for Random Forest
print("\n--- Random Forest Metrics (on Training Data) ---")
# Accuracy
print(f"Accuracy: {accuracy_score(y, y_pred_rf):.4f}")
# Precision
print(f"Precision: {precision_score(y, y_pred_rf, pos_label='Yes'):.4f}")
# Recall
print(f"Recall: {recall_score(y, y_pred_rf, pos_label='Yes'):.4f}")
# F1-Score
print(f"F1-Score: {f1_score(y, y_pred_rf, pos_label='Yes'):.4f}")

--- Logistic Regression Metrics (on Training Data) ---
Accuracy: 0.8062
Precision: 0.6611
Recall: 0.5554
F1-Score: 0.6037

--- Random Forest Metrics (on Training Data) ---
Accuracy: 0.8645
Precision: 0.7951
Recall: 0.6602
F1-Score: 0.7214


## **Findings and provide insights**


## Problem Definition:
The project addressed the critical business problem of customer churn in the telecommunications industry, aiming to build a predictive model to identify customers likely to churn and support targeted retention efforts.

##Data Preprocessing:
* The Telco Churn Dataset was loaded and inspected.
* Missing values in the 'TotalCharges' column (represented as ' ') were identified, converted to NaN, and handled by dropping the affected rows.
*  Features (X) and the target variable (y - Churn) were separated.
* Categorical and numerical features were identified.
* A preprocessing pipeline was built using Scikit-learn's ColumnTransformer to apply StandardScaler to numerical features and OneHotEncoder to categorical features, excluding the 'customerID'. This pipeline ensures consistent preprocessing for both training and future predictions.

## Model Development and Training:
* Logistic Regression and Random Forest classifiers were chosen as the initial models.
* Pipelines incorporating the preprocessing steps and the respective classifiers were created.
* The pipelines were trained on the preprocessed data. An initial error related to the 'customerID' column was resolved by explicitly dropping it from the feature set before training.

## Hyperparameter Tuning:
* GridSearchCV was employed to find the optimal hyperparameters for both models using 5-fold cross-validation and 'accuracy' as the scoring metric.
* Best parameters for Logistic Regression were found to be `{'classifier__C': 100}`.
* Best parameters for Random Forest were found to be `{'classifier__max_depth': 10, 'classifier__n_estimators': 200}`.

## Model Evaluation:
* The best-performing models from GridSearchCV were evaluated on the training data using Accuracy, Precision, Recall, and F1-Score.
* Evaluation Metrics:
Logistic Regression: Accuracy: 0.8062, Precision: 0.6611, Recall: 0.5554, F1-Score: 0.6037 - Random Forest: Accuracy: 0.8645, Precision: 0.7951, Recall: 0.6602, F1-Score: 0.7214

## Comparison and Insights:
* Based on the evaluation metrics calculated on the training data, the Random Forest model significantly outperformed the Logistic Regression model across all metrics (Accuracy, Precision, Recall, and F1-Score).
* The higher Accuracy indicates that the Random Forest model correctly classified a larger proportion of both churn and non-churn customers.
* The higher Precision suggests that when the Random Forest model predicts a customer will churn, it is more likely to be correct compared to the Logistic Regression model. This is important for targeting retention campaigns efficiently.
* The higher Recall indicates that the Random Forest model is better at identifying actual churn customers. This is crucial for minimizing missed opportunities to retain at-risk customers.
* The higher F1-Score, which is the harmonic mean of Precision and Recall, confirms that the Random Forest model provides a better balance between these two metrics.
*The Random Forest model's superior performance could be attributed to its ability to capture non-linear relationships and interactions between features, which a linear model like Logistic Regression might not effectively handle.

## Preferred Model:
* Based on the evaluation on the training data, the Random Forest model is preferred for predicting customer churn in this dataset due to its higher overall performance and better balance of precision and recall.

## Potential Next Steps and Improvements:
* **Evaluate on a separate test set:** The current evaluation was done on the training data. To get a more reliable estimate of the model's performance on unseen data, the pipeline should be evaluated on a dedicated test set that was held out from the beginning. This will help assess the model's generalization ability and detect potential overfitting.
* **Cross-validation for final evaluation:** Instead of just evaluating on a single train/test split, using cross-validation on the entire dataset (after initial data cleaning) can provide a more robust estimate of model performance.
* **Explore other models:** Investigate other classification algorithms such as Gradient Boosting (e.g., XGBoost, LightGBM), Support Vector Machines, or Neural Networks, which might capture different patterns in the data.
* **Feature Engineering:** Create new features from existing ones that might provide more predictive power (e.g., customer tenure categories, service bundles).
* **Address Class Imbalance:** Churn datasets often have imbalanced classes (fewer churn instances than non-churn). Techniques like oversampling (SMOTE), undersampling, or using class weights during training could improve the model's ability to predict the minority class (churn).
* **Advanced Hyperparameter Tuning:** Use more sophisticated tuning techniques like RandomizedSearchCV or Bayesian Optimization, or expand the hyperparameter grids for more extensive searching.
* **Interpretability:** While Random Forest provides good performance, understanding the feature importances can offer valuable business insights into why customers churn. Techniques like SHAP or LIME could be used for local interpretability.
* **Threshold Tuning:** The default classification threshold is 0.5. Tuning this threshold can optimize the balance between precision and recall based on the specific business requirements (e.g., is it more important to identify as many churners as possible (recall) or to be highly confident in the identified churners (precision)?).
* **Deployment Considerations:** Plan for deploying the exported pipeline for real-time or batch predictions.

## Model Export:
* The best-performing Random Forest pipeline was successfully exported using joblib, making it ready for deployment.

